## Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils import resample

## Load Dataset

In [ ]:
df = pd.read_csv('/content/data_ecommerce_customer_churn.csv')
print("Shape:", df.shape)
df.head()

## Exploratory Data Analysis

In [ ]:
print("=== Missing Values ===")
print(df.isnull().sum())
print()
print("=== Churn Distribution ===")
print(df['Churn'].value_counts())
print()
churn_pct = df['Churn'].value_counts(normalize=True) * 100
print(f"Churn rate: {churn_pct[1]:.1f}%")

## Preprocessing

In [ ]:
data = df.copy()

# 1. Encode kolom kategorikal
le_dict = {}  # simpan encoder per kolom biar bisa decode nanti
for col in data.select_dtypes(include='object').columns:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    le_dict[col] = le

# 2. Imputasi missing values dengan median (lebih robust dari mean)
num_cols = data.select_dtypes(include='float64').columns
imputer = SimpleImputer(strategy='median')
data[num_cols] = imputer.fit_transform(data[num_cols])

print("Missing values setelah imputasi:")
print(data.isnull().sum())
print()
print("Shape setelah preprocessing:", data.shape)

## Handle Imbalanced Data (Oversampling)

In [ ]:
X = data.drop('Churn', axis=1)
y = data['Churn']

print("Distribusi sebelum oversampling:")
print(y.value_counts())

# Oversampling kelas minoritas (Churn=1) pakai resample dari sklearn
df_majority = data[data['Churn'] == 0]
df_minority = data[data['Churn'] == 1]

df_minority_upsampled = resample(
    df_minority,
    replace=True,
    n_samples=len(df_majority),
    random_state=42
)

data_balanced = pd.concat([df_majority, df_minority_upsampled])
data_balanced = data_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

X = data_balanced.drop('Churn', axis=1)
y = data_balanced['Churn']

print()
print("Distribusi setelah oversampling:")
print(y.value_counts())

## Split & Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Train size: {X_train.shape[0]}")
print(f"Test size:  {X_test.shape[0]}")

## Cari K Terbaik

In [ ]:
acc_list = []

for k in range(1, 21):
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc_list.append(accuracy_score(y_test, pred))

best_k = acc_list.index(max(acc_list)) + 1
print(f"K terbaik: {best_k} dengan akurasi: {max(acc_list):.4f}")

plt.figure(figsize=(10, 5))
plt.plot(range(1, 21), acc_list, marker='o', color='steelblue', linewidth=2)
plt.axvline(x=best_k, color='red', linestyle='--', label=f'Best K={best_k}')
plt.xlabel('K')
plt.ylabel('Accuracy')
plt.title('KNN Accuracy untuk Berbagai Nilai K')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Training Model KNN

In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

print("=== Evaluasi Model ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=['Tidak Churn', 'Churn']))

## Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Tidak Churn', 'Churn'],
            yticklabels=['Tidak Churn', 'Churn'])
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()